# 03 — Baselines: ItemKNN and global popularity

Same date split, rows, scorer interface and evaluation as notebooks 01 / 02. **ItemKNN** (`implicit` cosine, item-item, K neighbours): history only; fitted on every train-window booking; cold rows (empty history) get the train-window first-booking popularity; ties broken by train popularity. **Global popularity**: every row gets the same ranking, the train-window booking share of each cluster (no history, no query; nothing to tune, nothing random). K is chosen on **autoregressive valid NDCG@10** with the one-SE rule by user (ties to the smaller K). Popularity reads no history, so its autoregressive and teacher-forced ranks must be identical (asserted).


In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / ("results_fast" if FAST else "results")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 14


In [2]:
def summary_tables(ranks: dict) -> dict:
    """Metrics per window x protocol x row set (unrounded)."""
    out = {}
    for w in ("valid", "test"):
        masks = row_set_masks(L[w])
        for proto in ("ar", "tf"):
            out[f"{w}_{proto}"] = {rs: metrics(ranks[f"{w}_{proto}"], m) for rs, m in masks.items()}
    return out


def save_ranks(name: str, ranks: dict) -> Path:
    RANKS.mkdir(parents=True, exist_ok=True)
    path = RANKS / f"ranks_{name}.npz"
    np.savez_compressed(path, **{k: v for k, v in ranks.items()}, **{f"event_{w}": rows["event"][i] for w, i in idx.items()})
    return path

In [3]:
from nbp.baselines.itemknn import fit_itemknn, itemknn_scorer, user_item_matrix
from nbp.baselines.popularity import fit_popularity, popularity_scorer
from nbp.eval.metrics import target_rank
from nbp.hybrid.fusion import one_se_pick
from nbp.models.train import git_hash

if FAST and TRAIN_CFG["min_delta"] is None:
    TRAIN_CFG["min_delta"] = json.loads((RES / "min_delta_calibration.json").read_text(encoding="utf-8"))["min_delta"]
assert TRAIN_CFG["min_delta"] is not None, "run notebook 00 first (min_delta calibration)"


def per_row(r, k=K_MON):
    return (r <= k) / np.log2(r + 1)


tr = np.flatnonzero(rows["window"] == TRAIN)
y_tr = rows["y"][tr].astype(np.int64)
pop_share = np.bincount(y_tr, minlength=N_ITEMS) / len(y_tr)
first = tr[rows["L"][tr] == 0]
pop_first_share = np.bincount(rows["y"][first], minlength=N_ITEMS) / len(first)
print("train-window bookings", len(tr), "| first bookings", len(first))

train-window bookings 2180488 | first bookings 664727


## ItemKNN

In [4]:
ui = user_item_matrix(rows["user"][tr], y_tr, N_ITEMS)
knn_valid, sims = {}, {}
for k in TRAIN_CFG["itemknn_k_grid"]:
    sims[k] = fit_itemknn(ui, k)
    sc = R.score_view(views["valid"], itemknn_scorer(sims[k], pop_share, pop_first_share, TRAIN_CFG["itemknn_tie_eps"]), autoregressive=True, batch=BATCH)
    knn_valid[k] = target_rank(sc.scores, Y["valid"])
    print("K", k, "AR valid ndcg@10", round(per_row(knn_valid[k]).mean(), 4))
K_PICK, k_best, k_elig = one_se_pick({k: per_row(r) for k, r in knn_valid.items()}, USERS["valid"], complexity=lambda k: k)
print("chosen K", K_PICK, "| best", k_best, "| within one SE", k_elig)
knn_sf = itemknn_scorer(sims[K_PICK], pop_share, pop_first_share, TRAIN_CFG["itemknn_tie_eps"])
r_knn = R.evaluate(knn_sf, views, Y, BATCH)
save_ranks("itemknn", r_knn)
h0 = {w: H[w] == 0 for w in ("valid", "test")}
assert all(np.array_equal(r_knn[f"{w}_ar"][h0[w]], r_knn[f"{w}_tf"][h0[w]]) for w in h0), "ItemKNN: h = 0 rows must not depend on the protocol"
write_json(RES / "itemknn.json", {"model": "ItemKNN", "k_chosen": K_PICK, "k_best_on_valid": k_best, "k_within_one_se": k_elig,
           "valid_ar_ndcg@10_grid": {str(k): float(per_row(r).mean()) for k, r in knn_valid.items()},
           "fit_rows": "every train-window booking", "cold_rows": "train-window first-booking popularity",
           "git_commit": git_hash(ROOT), "split": SPLIT, "results": summary_tables(r_knn)})
t = summary_tables(r_knn)["test_ar"]
print(f"ItemKNN test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.013988733291625977 seconds
  warnings.warn(


K 5 AR valid ndcg@10 0.1741


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.008729219436645508 seconds
  warnings.warn(


K 10 AR valid ndcg@10 0.1756


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.008893489837646484 seconds
  warnings.warn(


...

C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.0 seconds
  warnings.warn(


K 50 AR valid ndcg@10 0.1772


C:\workspace\PROJECTS\next-best-product-recommendation\.venv\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.017873764038085938 seconds
  warnings.warn(


K 100 AR valid ndcg@10 0.177
chosen K 50 | best 50 | within one SE [50]


ItemKNN test AR all ndcg@10 0.1825 warm 0.1990 cold 0.1118


## Global popularity

In [5]:
share = fit_popularity(y_tr, N_ITEMS)  # train-window booking share per model id
r_pop = R.evaluate(popularity_scorer(share), views, Y, BATCH)
for w in ("valid", "test"):
    assert np.array_equal(r_pop[f"{w}_ar"], r_pop[f"{w}_tf"]), "popularity must not depend on the protocol"
save_ranks("popularity", r_pop)
top5 = np.argsort(-share)[:5] - 1  # hotel_cluster ids of the five most booked train-window clusters
write_json(RES / "popularity.json", {"model": "Global popularity", "fit_rows": "every train-window booking",
           "ranking": "train-window booking share of each cluster, the same list for every row",
           "top5_clusters": [int(c) for c in top5], "top5_shares": [float(share[c + 1]) for c in top5],
           "git_commit": git_hash(ROOT), "split": SPLIT, "results": summary_tables(r_pop)})
t = summary_tables(r_pop)["test_ar"]
print(f"popularity test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

popularity test AR all ndcg@10 0.1225 warm 0.1250 cold 0.1114
